# 02 - Train a DQN Model Offline

This notebook shows the offline training workflow in Mouse Core:

1. Build a held-out `GroupEnv` for greedy evaluation during training.
2. Load previously collected `Datastore` streams from the Hub.
3. Build a `DataLoader` that samples fixed-length sequences from those streams.
4. Assemble a `Model` from an embedder, a backbone, and an action-value head.
5. Train for `NUM_CYCLES` cycles (`TRAIN_STEPS` optimizer updates then `EVAL_STEPS` eval tasks each) and save with `push_model_to_hub`.

The dataset comes from the collection notebook, but the same Mouse Core pieces apply to any sequential environment data stored as step dictionaries.

Each cycle runs **greedy eval** on the held-out `mouse-gym` environment group (not the Hub replay streams) and prints mean task score.


In [ ]:
import torch
import numpy as np

import procedural_frozenlake  # noqa: F401 — registers Procedural-FrozenLake-v1
from mouse_gym import EnvConfig, make_group_env
from mouse_core.models.kv_policy import (
    cache_needs_rebuild,
    rebuild_starts,
    resolve_cache_bounds,
)

from mouse_core.data import (
    DataLoader,
    Augmenter,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective
from mouse_core.models import Model, preferred_dtype, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import NumericEmbedder
from mouse_core.models.heads import DiscreteActionValueHead


DATASET_ID = "mouse-example-dataset"          # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-offline"      # Hugging Face model repo for push_model_to_hub
MAX_ACTIONS = 4                               # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
SEQUENCE_LENGTH = 512                         # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step
NUM_CYCLES = 20                              # outer train/eval cycles
TRAIN_STEPS = 1000                           # optimizer updates per cycle (passed to run_train)
EVAL_STEPS = 1                               # tasks per eval env per cycle (passed to run_eval)
EVAL_NUM_ENVS = 4                             # separate eval env streams (not used for replay)
EPISODES_PER_TASK = 20                        # environment-specific task length
MAX_EPISODE_STEPS = 30                             # environment-specific episode limit
EVAL_SEED_OFFSET = 1_000_000                  # held-out env seed stream (far from train)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Build Environment

Offline training reads Hub replay only, but progress is measured on a **separate** held-out `GroupEnv`. Use the same `EnvConfig` / `make_group_env` pattern as data collection; offset seeds with `EVAL_SEED_OFFSET` so eval maps are not the collection set.

`make_group_env` steps all configured instances together. Keep row fields (`action`, `observation`, `reward`, `done`) consistent with the model modalities and the DQN objective.


In [ ]:
eval_configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"eval_frozenlake_{i}",
        reset_seed=i + EVAL_SEED_OFFSET,
        episodes_per_task=EPISODES_PER_TASK,
        task_reset_options={"regenerate_map": True},
        kwargs={
            "width": 8,
            "height": 8,
            "max_episode_steps": MAX_EPISODE_STEPS,
            "map_seed": i + EVAL_SEED_OFFSET,
            "slippery_success_rate": 1.0,
            "permute_obs": True,
            "permute_actions": True,
        },
    )
    for i in range(EVAL_NUM_ENVS)
]

eval_env = make_group_env(eval_configs)


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)

## DataLoader And Augmentation

`DataLoader` samples contiguous windows up to `sequence_length` (a max) from one or more datastores. Each sequence may be shorter than the max depending on where the window starts in the store.

`next_batch()` returns a prepared `TokenBatch` when `preparer=` is set, or a ragged `list[list[dict]]` for debugging.

`Augmenter` can transform fields as batches are sampled. A modality spec names a row field and an augmentation type, such as `discrete`, `linear`, or `image`. Use `keep_fields` for values that should pass through unchanged because an objective still needs them. Segment IDs are returned separately from row dicts and are not part of `keep_fields`.


In [ ]:
augmenter = Augmenter(
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
    keep_fields=["action", "observation", "reward", "done"],
)

loader = DataLoader(stores=stores, sequence_length=SEQUENCE_LENGTH, batch_size=BATCH_SIZE, prefetch=4, num_workers=0, augmenter=augmenter)

## Build The Model

A Mouse Core `Model` has three main pieces:

- `NumericEmbedder` converts fields from each step dictionary into model tokens.
- `Qwen3Backbone` processes those tokens with a transformer backbone.
- `DiscreteActionValueHead` predicts one value per discrete action.

The backbone exposes `hidden_dim`, and the embedder and head use that same value so the pieces connect cleanly.

`NumericEmbedder` modality types used here:

- `discrete` for integer IDs such as actions, observations, and done codes.
- `rff` for scalar numeric values such as rewards.
- `learnable` when you want extra learned tokens that are not tied to a row field.

`Model(...)` wraps the pieces behind a single forward call that returns predictions, objective data, and an optional cache.


In [ ]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = NumericEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "tokens": 1
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1
        },
    ],
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device=device, dtype=preferred_dtype(device))
model.backbone.model.compile()  # optional compile step for faster repeated forwards
print(model)


## Evaluation Phase

`run_eval` rolls out the current policy greedily on `eval_env`. It does not append to train replay. Row **contexts persist** across calls so evaluation continues mid-task; the **KV cache does not** and is rebuilt from those contexts at the start of each call. Each outer cycle passes `num_steps=EVAL_STEPS` (tasks per eval env).


In [ ]:
eval_contexts = [[] for _ in eval_env.names]

def run_eval(
    *,
    model,
    env,
    contexts: list,
    num_steps: int,
    max_cache: int = 512,
    start_cache=None,
    temperature: float = 0.0,
):
    """Greedy task-budget eval on a GroupEnv (does not append to train replay).

    ``num_steps`` is the number of tasks to complete per env stream.
    ``contexts`` persist across calls (mutated in place); the KV cache does not
    and is rebuilt from ``contexts`` at the start of each call.
    """
    model.eval()
    max_cache, start_cache = resolve_cache_bounds(max_cache, start_cache)
    n = len(env.names)
    if len(contexts) != n:
        raise ValueError(f"contexts has {len(contexts)} streams but env has {n}")
    kv_cache = None
    cached_starts = np.zeros(n, dtype=np.int64)
    cached_ends = np.zeros(n, dtype=np.int64)
    context_start = np.zeros(n, dtype=np.int64)
    eval_task_scores = [[] for _ in range(n)]
    inputs = None
    outputs = None
    env.metrics.clear()
    num_actions = env.action_space.spaces[0].n

    def _act_from_contexts(*, rebuild: bool) -> list[dict]:
        nonlocal kv_cache, cached_starts, cached_ends
        ends = np.array([len(c) for c in contexts], dtype=np.int64)
        need_rebuild = rebuild or cache_needs_rebuild(
            has_cache=kv_cache is not None,
            cached_starts=cached_starts,
            cached_ends=cached_ends,
            ends=ends,
            context_start=context_start,
            max_cache=max_cache,
            batch_complete=True,
        )
        with torch.no_grad():
            if need_rebuild:
                starts = rebuild_starts(
                    ends=ends,
                    context_start=context_start,
                    start_cache=start_cache,
                    max_cache=max_cache,
                )
                batch = [contexts[i][int(starts[i]) : int(ends[i])] for i in range(n)]
                predictions, _, kv_cache = model(batch, use_cache=True)
                cached_starts = starts
                cached_ends = ends.copy()
            else:
                batch = [
                    contexts[i][int(cached_ends[i]) : int(ends[i])] for i in range(n)
                ]
                predictions, _, kv_cache = model(batch, cache=kv_cache, use_cache=True)
                cached_ends = ends.copy()
            actions = model.get_action(
                predictions, temperature=temperature, num_actions=num_actions
            )
        random_inputs = env.sample_random_input()
        return [
            {"action": action} if contexts[i] else random_inputs[i]
            for i, action in enumerate(actions.cpu().numpy())
        ]

    while min(len(scores) for scores in eval_task_scores) < num_steps:
        if outputs is None:
            # Start of this call: rebuild KV from persisted contexts (or random if empty).
            if any(contexts):
                inputs = _act_from_contexts(rebuild=True)
            else:
                inputs = env.sample_random_input()
        else:
            task_ended = False
            for i, (inp, out) in enumerate(zip(inputs, outputs)):
                row = {**inp, **out}
                row.pop("info", None)
                contexts[i].append(row)
                if int(row["done"]) >= 3:
                    contexts[i] = []
                    context_start[i] = 0
                    task_ended = True
            inputs = _act_from_contexts(rebuild=task_ended)
        outputs = env.step(inputs)
        for i, out in enumerate(outputs):
            if int(out["done"]) >= 3:
                eval_task_scores[i].append(float(env.metrics.task_cum_rewards[i][-1]))

    scores_per_env = [scores[:num_steps] for scores in eval_task_scores]
    flat_scores = [s for scores in scores_per_env for s in scores]
    mean_task_score = (
        sum(flat_scores) / len(flat_scores) if flat_scores else float("nan")
    )
    return {
        "mean_task_score": mean_task_score,
        "task_scores": flat_scores,
        "scores_per_env": scores_per_env,
        "n_tasks": len(flat_scores),
    }


## Training Phase

The offline loop is intentionally small. Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`, then scores the held-out env with `run_eval`. Mouse Core abstractions do most of the work:

1. `loader.next_batch()` samples ragged step windows (up to `SEQUENCE_LENGTH`).
2. `model(batch)` embeds the dictionaries, runs the backbone with per-sequence causal attention/RoPE, and produces flat per-step head predictions.
3. `objective(objective_data, predictions)` computes the DQN loss and metrics.
4. The optimizer updates model weights.
5. `model.polyak_update(...)` updates target-network weights used by the objective.

`DqnObjective` interprets the integer `done` field through separate discount factors for normal steps, episode boundaries, and task boundaries. Set those gammas to match the semantics of your environment data.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-05, weight_decay=0.0, betas=(0.9, 0.95), eps=1e-08)
objective = DqnObjective(gamma_step=1.0, gamma_episode_terminal=1.0, gamma_episode_truncated=1.0, gamma_task_terminal=0.0, gamma_task_truncated=0.0, tau=0.0005)

def run_train(*, model: Model, optimizer: torch.optim.Optimizer, objective: DqnObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        batch = loader.next_batch()
        predictions, objective_data, _ = model(batch)
        loss, metrics = objective(objective_data, predictions)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.polyak_update(action_value_tau=objective.tau)
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`, then `run_eval(num_steps=EVAL_STEPS)`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(model=model, optimizer=optimizer, objective=objective, loader=loader, num_steps=TRAIN_STEPS)
    print(f"cycle={cycle} train  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}")
    stats = run_eval(num_steps=EVAL_STEPS, max_cache=SEQUENCE_LENGTH, model=model, env=eval_env, contexts=eval_contexts)
    print(f"cycle={cycle} eval   mean_task_score={stats['mean_task_score']:.2f}/{EPISODES_PER_TASK}  n_tasks={stats['n_tasks']}")
loader.close()
eval_env.close()


## Push To The Hub

`push_model_to_hub` saves the model architecture and weights together. Later, `load_model` can reconstruct the full `Model` without repeating the embedder, backbone, and head definitions.


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model=model, repo_id=MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")